# Ch.11 — Advanced Agentic Patterns

> **Source notes:** `README.md`

Single-pass LLM calls fail on edge cases. This notebook demonstrates four production agentic patterns that trade tokens for reliability:

1. **Reflection** — Draft → Critique → Revise (handle contradictions)
2. **Debate** — Multi-agent consensus (resolve policy ambiguity)
3. **Hierarchical Orchestration** — Planner → Workers → Verifier (complex multi-step tasks)
4. **Tool Selection** — Meta-agent routing with fallback chains (optimize cost/latency)

**Running example:** PizzaBot v2.0 handling contradictory orders, pricing conflicts, and catering requests.

**Setup:** All examples use mock API mode by default (no OpenAI key required). Set `USE_MOCK_API = False` for real LLM calls.

In [ ]:
def get_llm():
    """
    TODO #1: Implement `get_llm()`.

    Steps:
    1. Set up: ─
    2. Call `run()` to produce the result
    3. Process data
    4. Compute `USE_MOCK_API`
    5. Define helper function
    6. Define helper function `get_llm()`
    7. Process data

    Hint:
    # implement using the APIs described above

    Returns: self.responses.get("critique", {
    """
    raise NotImplementedError("TODO: implement get_llm()")

## 1 · Pattern 1: Reflection Loop — Draft → Critique → Revise

**Scenario:** "Gluten-free, dairy-free pizza with extra cheese" (contradiction)

**How it works:**
```
Step 1: Draft response (single-pass attempt)
Step 2: Self-critique (detect contradiction)
Step 3: Revise response (suggest vegan cheese alternative)
```

**Token economics:**
- Draft: 150 tokens
- Critique: 100 tokens
- Revision: 180 tokens
- **Total: 430 tokens** (2.9× single-pass) → **6× error reduction**

In mock mode, responses are hardcoded to demonstrate the pattern flow.

In [ ]:
def reflection_loop(query: str):
    """
    TODO #2: Implement `reflection_loop()`.

    Steps:
    1. Set up: ─
    2. Define helper function `reflection_loop()`
    3. Call `set_response()` to produce the result
    4. Call `invoke()` to produce the result
    5. Call `set_response()` to produce the result
    6. Call `invoke()` to produce the result
    7. Call `set_response()` to produce the result
    8. Call `invoke()` to produce the result
    9. Process data
    10. Compute `query`
    11. Call `Draft()` to produce the result
    12. Process data
    13. Call `better()` to produce the result

    Hint:
    draft_response = llm.invoke(???)
    critique_response = llm.invoke(???)
    revised_response = llm.invoke(???)

    Returns: {
    """
    raise NotImplementedError("TODO: implement reflection_loop()")

## 2 · Pattern 2: Debate — Multi-Agent Consensus

**Scenario:** Pricing conflict — customer has $5 coupon + 10% loyalty + 15% promo active. Which applies?

**How it works:**
```
Agent 1 (generous): Stack all discounts → $10.75
Agent 2 (strict): Apply best only → $15.30
Judge (policy check): Queries RAG policy doc → "one discount per order" → Agent 2 wins
```

**Token economics:**
- Agent 1 proposal: 120 tokens
- Agent 2 proposal: 110 tokens
- Agent 1 rebuttal: 130 tokens
- Agent 2 rebuttal: 120 tokens
- Judge + policy lookup: 420 tokens
- **Total: 900 tokens** (6× single-pass) → **8× error reduction**

Debate prevents incorrect discount stacking that costs the business money.

In [ ]:
def multi_agent_debate(query: str, num_rounds: int = 2):
    """
    TODO #3: Implement `multi_agent_debate()`.

    Steps:
    1. Set up: ─
    2. Define helper function `multi_agent_debate()`
    3. Call `set_response()` to produce the result
    4. Call `1()` to produce the result
    5. Call `set_response()` to produce the result
    6. Call `2()` to produce the result
    7. Call `set_response()` to produce the result
    8. Call `invoke()` to produce the result
    9. Call `set_response()` to produce the result
    10. Call `invoke()` to produce the result
    11. Call `set_response()` to produce the result
    12. Call `doc()` to produce the result
    13. Process data
    14. Compute `query`
    15. Process data
    16. Call `RULING()` to produce the result
    17. Call `better()` to produce the result

    Hint:
    agent1_round1 = llm.invoke(???)
    agent2_round1 = llm.invoke(???)
    agent1_round2 = llm.invoke(???)
    agent2_round2 = llm.invoke(???)

    Returns: {
    """
    raise NotImplementedError("TODO: implement multi_agent_debate()")

## 3 · Pattern 3: Hierarchical Orchestration — Planner → Workers → Verifier

**Scenario:** Catering order — 15 pizzas, 3 delivery times (11am, 12pm, 1pm), $200 budget

**How it works:**
```
Planner: Split into 3 batches (5 pizzas each)
Worker 1: Process 11am batch → 5 Margherita = $60
Worker 2: Process 12pm batch → 5 Pepperoni = $70
Worker 3: Process 1pm batch → 3 Veggie + 2 Margherita = $56
Verifier: Check total $186 < $200 , all times feasible
```

**Token economics:**
- Planner: 200 tokens
- 3 workers (parallel): 3 × 150 = 450 tokens
- Verifier: 100 tokens
- **Total: 750 tokens** (5× single-pass) → **15× error reduction**

Hierarchical decomposition prevents single-pass from missing valid solutions.

In [ ]:
def hierarchical_orchestration(query: str):
    """
    TODO #4: Implement `hierarchical_orchestration()`.

    Steps:
    1. Set up: ─
    2. Define helper function `hierarchical_orchestration()`
    3. Call `set_response()` to produce the result
    4. Call `invoke()` to produce the result
    5. Call `parallel()` to produce the result
    6. Call `set_response()` to produce the result
    7. Call `1()` to produce the result
    8. Call `set_response()` to produce the result
    9. Call `2()` to produce the result
    10. Call `set_response()` to produce the result
    11. Call `3()` to produce the result
    12. Call `set_response()` to produce the result
    13. Aggregate / merge data
    14. Process data
    15. Compute `query` using `batches()`
    16. Process data
    17. Call `better()` to produce the result

    Hint:
    plan = llm.invoke(???)
    worker1 = llm.invoke(???)
    worker2 = llm.invoke(???)
    worker3 = llm.invoke(???)

    Returns: {
    """
    raise NotImplementedError("TODO: implement hierarchical_orchestration()")

## 4 · Pattern 4: Tool Selection with Fallback Chains

**Scenario:** Check pizza inventory with 3 data sources: cache (free, stale), DB (moderate), API (slow, authoritative)

**Strategies:**

1. **Rule-based fallback:** Cache → DB → API → escalate
2. **Cost-optimized:** If time-sensitive → skip cache; else use cache
3. **Meta-agent:** LLM decides which tool based on query semantics

**Token economics:**
- Meta-agent call: 80 tokens
- Tool execution: +0 (cache), +0 (DB), +0 (API)
- **Total: 80-230 tokens** (1.5× single-pass) → **4× error reduction**

Meta-agent routing is worth it when you have >3 tools and complex routing logic.

In [ ]:
def tool_cache():
    """
    TODO #5: Implement `tool_cache()`.

    Steps:
    1. Set up: ─
    2. Process data
    3. Define helper function
    4. Define helper function `tool_cache()`
    5. Define helper function `tool_database()`
    6. Call `sleep()` to produce the result
    7. Define helper function `tool_external_api()`
    8. Call `sleep()` to produce the result
    9. Define helper function `strategy_fallback_chain()`
    10. Process data
    11. Call `tool_func()` to produce the result
    12. Process data
    13. Call `failed()` to produce the result
    14. Process data
    15. Define helper function `strategy_cost_optimized()`
    16. Call `cache()` to produce the result
    17. Define helper function `strategy_meta_agent()`
    18. Call `get_llm()` to produce the result
    19. Call `set_response()` to produce the result
    20. Process data
    21. Call `cache()` to produce the result
    22. Process data
    23. Call `invoke()` to produce the result
    24. Call `lower()` to produce the result
    25. Process data
    26. Call `optimized()` to produce the result
    27. Call `1234()` to produce the result
    28. Process data

    Hint:
    meta_response = meta_llm.invoke(???)

    Returns: ToolResult(
    """
    raise NotImplementedError("TODO: implement tool_cache()")

def tool_database(simulate_timeout: bool = False):
    """
    TODO #5: Implement `tool_database()`.

    Steps:
    1. Set up: ─
    2. Process data
    3. Define helper function
    4. Define helper function `tool_cache()`
    5. Define helper function `tool_database()`
    6. Call `sleep()` to produce the result
    7. Define helper function `tool_external_api()`
    8. Call `sleep()` to produce the result
    9. Define helper function `strategy_fallback_chain()`
    10. Process data
    11. Call `tool_func()` to produce the result
    12. Process data
    13. Call `failed()` to produce the result
    14. Process data
    15. Define helper function `strategy_cost_optimized()`
    16. Call `cache()` to produce the result
    17. Define helper function `strategy_meta_agent()`
    18. Call `get_llm()` to produce the result
    19. Call `set_response()` to produce the result
    20. Process data
    21. Call `cache()` to produce the result
    22. Process data
    23. Call `invoke()` to produce the result
    24. Call `lower()` to produce the result
    25. Process data
    26. Call `optimized()` to produce the result
    27. Call `1234()` to produce the result
    28. Process data

    Hint:
    meta_response = meta_llm.invoke(???)

    Returns: ToolResult(
    """
    raise NotImplementedError("TODO: implement tool_database()")

def tool_external_api(simulate_timeout: bool = False):
    """
    TODO #5: Implement `tool_external_api()`.

    Steps:
    1. Set up: ─
    2. Process data
    3. Define helper function
    4. Define helper function `tool_cache()`
    5. Define helper function `tool_database()`
    6. Call `sleep()` to produce the result
    7. Define helper function `tool_external_api()`
    8. Call `sleep()` to produce the result
    9. Define helper function `strategy_fallback_chain()`
    10. Process data
    11. Call `tool_func()` to produce the result
    12. Process data
    13. Call `failed()` to produce the result
    14. Process data
    15. Define helper function `strategy_cost_optimized()`
    16. Call `cache()` to produce the result
    17. Define helper function `strategy_meta_agent()`
    18. Call `get_llm()` to produce the result
    19. Call `set_response()` to produce the result
    20. Process data
    21. Call `cache()` to produce the result
    22. Process data
    23. Call `invoke()` to produce the result
    24. Call `lower()` to produce the result
    25. Process data
    26. Call `optimized()` to produce the result
    27. Call `1234()` to produce the result
    28. Process data

    Hint:
    meta_response = meta_llm.invoke(???)

    Returns: ToolResult(
    """
    raise NotImplementedError("TODO: implement tool_external_api()")

def strategy_fallback_chain(simulate_failures: bool = False):
    """
    TODO #5: Implement `strategy_fallback_chain()`.

    Steps:
    1. Set up: ─
    2. Process data
    3. Define helper function
    4. Define helper function `tool_cache()`
    5. Define helper function `tool_database()`
    6. Call `sleep()` to produce the result
    7. Define helper function `tool_external_api()`
    8. Call `sleep()` to produce the result
    9. Define helper function `strategy_fallback_chain()`
    10. Process data
    11. Call `tool_func()` to produce the result
    12. Process data
    13. Call `failed()` to produce the result
    14. Process data
    15. Define helper function `strategy_cost_optimized()`
    16. Call `cache()` to produce the result
    17. Define helper function `strategy_meta_agent()`
    18. Call `get_llm()` to produce the result
    19. Call `set_response()` to produce the result
    20. Process data
    21. Call `cache()` to produce the result
    22. Process data
    23. Call `invoke()` to produce the result
    24. Call `lower()` to produce the result
    25. Process data
    26. Call `optimized()` to produce the result
    27. Call `1234()` to produce the result
    28. Process data

    Hint:
    meta_response = meta_llm.invoke(???)

    Returns: ToolResult(
    """
    raise NotImplementedError("TODO: implement strategy_fallback_chain()")

def strategy_cost_optimized(time_sensitive: bool = False):
    """
    TODO #5: Implement `strategy_cost_optimized()`.

    Steps:
    1. Set up: ─
    2. Process data
    3. Define helper function
    4. Define helper function `tool_cache()`
    5. Define helper function `tool_database()`
    6. Call `sleep()` to produce the result
    7. Define helper function `tool_external_api()`
    8. Call `sleep()` to produce the result
    9. Define helper function `strategy_fallback_chain()`
    10. Process data
    11. Call `tool_func()` to produce the result
    12. Process data
    13. Call `failed()` to produce the result
    14. Process data
    15. Define helper function `strategy_cost_optimized()`
    16. Call `cache()` to produce the result
    17. Define helper function `strategy_meta_agent()`
    18. Call `get_llm()` to produce the result
    19. Call `set_response()` to produce the result
    20. Process data
    21. Call `cache()` to produce the result
    22. Process data
    23. Call `invoke()` to produce the result
    24. Call `lower()` to produce the result
    25. Process data
    26. Call `optimized()` to produce the result
    27. Call `1234()` to produce the result
    28. Process data

    Hint:
    meta_response = meta_llm.invoke(???)

    Returns: ToolResult(
    """
    raise NotImplementedError("TODO: implement strategy_cost_optimized()")

def strategy_meta_agent(query: str):
    """
    TODO #5: Implement `strategy_meta_agent()`.

    Steps:
    1. Set up: ─
    2. Process data
    3. Define helper function
    4. Define helper function `tool_cache()`
    5. Define helper function `tool_database()`
    6. Call `sleep()` to produce the result
    7. Define helper function `tool_external_api()`
    8. Call `sleep()` to produce the result
    9. Define helper function `strategy_fallback_chain()`
    10. Process data
    11. Call `tool_func()` to produce the result
    12. Process data
    13. Call `failed()` to produce the result
    14. Process data
    15. Define helper function `strategy_cost_optimized()`
    16. Call `cache()` to produce the result
    17. Define helper function `strategy_meta_agent()`
    18. Call `get_llm()` to produce the result
    19. Call `set_response()` to produce the result
    20. Process data
    21. Call `cache()` to produce the result
    22. Process data
    23. Call `invoke()` to produce the result
    24. Call `lower()` to produce the result
    25. Process data
    26. Call `optimized()` to produce the result
    27. Call `1234()` to produce the result
    28. Process data

    Hint:
    meta_response = meta_llm.invoke(???)

    Returns: ToolResult(
    """
    raise NotImplementedError("TODO: implement strategy_meta_agent()")

## 5 · Error Recovery with Exponential Backoff

**Problem:** Tools fail (timeout, rate limit, transient errors).

**Naive approach:** Single retry → still fails → show error to user

**Production approach:**
```
1. Retry with exponential backoff (1s, 2s, 4s, 8s)
2. Fall back to degraded service (cached/stale data)
3. Escalate to human only after all retries exhausted
```

This pattern is critical for production reliability.

In [ ]:
def unreliable_tool(attempt: int):
    """
    TODO #6: Implement `unreliable_tool()`.

    Steps:
    1. Set up: ─
    2. Process data
    3. Define helper function `unreliable_tool()`
    4. Define helper function `retry_with_backoff()`
    5. Process data
    6. Call `tool_func()` to produce the result
    7. Call `attempt()` to produce the result
    8. Call `data()` to produce the result
    9. Call `tool_cache()` to produce the result
    10. Process data
    11. Compute `result`
    12. Call `service()` to produce the result

    Hint:
    # implement using the APIs described above

    Returns: ToolResult(success=False, data=None, ...
    """
    raise NotImplementedError("TODO: implement unreliable_tool()")

def retry_with_backoff(tool_func, max_retries=4):
    """
    TODO #6: Implement `retry_with_backoff()`.

    Steps:
    1. Set up: ─
    2. Process data
    3. Define helper function `unreliable_tool()`
    4. Define helper function `retry_with_backoff()`
    5. Process data
    6. Call `tool_func()` to produce the result
    7. Call `attempt()` to produce the result
    8. Call `data()` to produce the result
    9. Call `tool_cache()` to produce the result
    10. Process data
    11. Compute `result`
    12. Call `service()` to produce the result

    Hint:
    # implement using the APIs described above

    Returns: ToolResult(success=False, data=None, ...
    """
    raise NotImplementedError("TODO: implement retry_with_backoff()")

## 6 · Cost vs Error Reduction Comparison

Now let's compare all four patterns quantitatively:

| Pattern | Tokens | Cost (GPT-4o-mini) | Latency | Error Reduction |
|---------|--------|-------------------|---------|-----------------|
| **Single-pass** | 150 | $0.000023 | 0.5s | baseline (8% error) |
| **Reflection** | 430 | $0.000065 | 1.5s | 6× better (1.2% error) |
| **Debate** | 900 | $0.000135 | 3.0s | 8× better (1.0% error) |
| **Hierarchical** | 750 | $0.000113 | 2.5s | 15× better (0.5% error) |
| **Tool Selection** | 230 | $0.000035 | 0.8s | 4× better (2.0% error) |

**Key insight:** You're trading 2-6× token cost for 4-15× error reduction. For high-stakes production use cases (customer orders, financial transactions), this is a winning trade.

In [ ]:
# TODO: Implement this cell
#  (─)
#
# Steps:
# 1. Set up: ─
# 2. Process data
# 3. Compute `patterns`
# 4. Plot results -- call `use()`
# 5. Plot results -- call `set_facecolor()`
# 6. Plot results -- call `bar()`
# 7. Plot results -- call `scatter()`
# 8. Call `set_xlabel()` to produce the result
# 9. Plot results -- call `barh()`
# 10. Plot results -- call `tight_layout()`
# 11. Process data
# 12. Call `detection()` to produce the result
#
# Hint:
#    axes = plt.subplots(???)
#    bars1 = ax1.bar(???)
#    bars2 = ax2.bar(???)
#    bars4 = ax4.barh(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ──────────────────────────────────────────────────────────────
# Cost vs Error Reduction Analysis
# ──────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import numpy as np

# Pattern comparison data
patterns = ["Single-pass", "Reflection", "Debate", "Hierarchical", "Tool Selection"]
tokens = [150, 430, 900, 750, 230]
costs = [t * 0.15 / 1_000_000 for t in tokens]  # GPT-4o-mini pricing
latencies = [0.5, 1.5, 3.0, 2.5, 0.8]
error_rates = [8.0, 1.2, 1.0, 0.5, 2.0]  # percentage
error_reduction = [1.0, 6.7, 8.0, 16.0, 4.0]  # relative to single-pass

# Set dark theme
plt.style.use('dark_background')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#1a1a2e')

for ax in axes.flat:
    ax.set_facecolor('#1a1a2e')

# Plot 1: Token cost comparison
ax1 = axes[0, 0]
bars1 = ax1.bar(patterns, tokens, color=['#666', '#4a9eff', '#ff6b6b', '#51cf66', '#ffd93d'])
ax1.set_ylabel('Tokens per conversation', fontsize=11)
ax1.set_title('Token Cost by Pattern', fontsize=13, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
for i, (bar, val) in enumerate(zip(bars1, tokens)):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
             f'{val}', ha='center', va='bottom', fontsize=10)

# Plot 2: Error reduction
ax2 = axes[0, 1]
bars2 = ax2.bar(patterns, error_rates, color=['#ff6b6b', '#51cf66', '#51cf66', '#51cf66', '#4a9eff'])
ax2.set_ylabel('Error rate (%)', fontsize=11)
ax2.set_title('Error Rate by Pattern', fontsize=13, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
ax2.axhline(y=1.0, color='#ffd93d', linestyle='--', linewidth=1, alpha=0.5, label='Production target (<1%)')
ax2.legend(fontsize=9)
for i, (bar, val) in enumerate(zip(bars2, error_rates)):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, 
             f'{val}%', ha='center', va='bottom', fontsize=10)

# Plot 3: Cost vs error reduction trade-off
ax3 = axes[1, 0]
colors_scatter = ['#666', '#4a9eff', '#ff6b6b', '#51cf66', '#ffd93d']
for i, pattern in enumerate(patterns):
    ax3.scatter(costs[i] * 1_000_000, error_reduction[i], 
                s=200, color=colors_scatter[i], alpha=0.8, edgecolors='white', linewidth=2)
    ax3.annotate(pattern, (costs[i] * 1_000_000, error_reduction[i]), 
                 xytext=(5, 5), textcoords='offset points', fontsize=9)

ax3.set_xlabel('Cost per conversation ($, ×10⁶)', fontsize=11)
ax3.set_ylabel('Error reduction (× vs single-pass)', fontsize=11)
ax3.set_title('Cost vs Error Reduction Trade-off', fontsize=13, fontweight='bold')
ax3.grid(alpha=0.2)

# Plot 4: Latency comparison
ax4 = axes[1, 1]
bars4 = ax4.barh(patterns, latencies, color=['#666', '#4a9eff', '#ff6b6b', '#51cf66', '#ffd93d'])
ax4.set_xlabel('Latency (seconds)', fontsize=11)
ax4.set_title('Latency by Pattern', fontsize=13, fontweight='bold')
ax4.axvline(x=15.0, color='#ffd93d', linestyle='--', linewidth=1, alpha=0.5, label='Production limit (15s)')
ax4.legend(fontsize=9, loc='lower right')
for i, (bar, val) in enumerate(zip(bars4, latencies)):
    ax4.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
             f'{val}s', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('img/pattern_comparison.png', dpi=150, facecolor='#1a1a2e')
print("✓ Cost comparison chart saved to img/pattern_comparison.png")
plt.show()

# Summary table
print("\n" + "=" * 90)
print("PATTERN COMPARISON SUMMARY")
print("=" * 90)
print(f"{'Pattern':<18} {'Tokens':>8} {'Cost':>12} {'Latency':>10} {'Error':>8} {'Reduction':>10}")
print("-" * 90)
for i, pattern in enumerate(patterns):
    print(f"{pattern:<18} {tokens[i]:>8} ${costs[i]:>11.6f} {latencies[i]:>9.1f}s {error_rates[i]:>7.1f}% {error_reduction[i]:>9.1f}×")

print("\n" + "=" * 90)
print("KEY TAKEAWAYS")
print("=" * 90)
print("1. Reflection: Best ROI for contradiction detection (6.7× error reduction, 2.9× cost)")
print("2. Debate: Worth it for high-stakes decisions (pricing, compliance)")
print("3. Hierarchical: Essential for complex multi-step tasks (15× error reduction)")
print("4. Tool Selection: Cheap insurance for tool reliability (4× error reduction, 1.5× cost)")
print("\nProduction rule: Use reflection by default, hierarchical for complex orders")

## 7 · When to Use Each Pattern — Decision Function

How do you decide which pattern to use for a given query?

Here's a decision tree function that recommends the right pattern based on query characteristics:

```
if has_contradiction(query):
 return REFLECTION
elif is_high_stakes(query):
 return DEBATE
elif is_multi_step_complex(query):
 return HIERARCHICAL
elif needs_multiple_tools(query):
 return TOOL_SELECTION
else:
 return SINGLE_PASS
```

Let's test it on real scenarios.

In [ ]:
def recommend_pattern(query: str):
    """
    TODO #8: Implement `recommend_pattern()`.

    Steps:
    1. Set up: ─
    2. Define helper function `recommend_pattern()`
    3. Call `contradictions()` to produce the result
    4. Call `queries()` to produce the result
    5. Call `complex()` to produce the result
    6. Call `needed()` to produce the result
    7. Process data
    8. Compute `test_queries`
    9. Process data
    10. Call `distribution()` to produce the result

    Hint:
    # implement using the APIs described above

    Returns: {
    """
    raise NotImplementedError("TODO: implement recommend_pattern()")

## 8 · Production Deployment with LangGraph State Machine

In production, these patterns are orchestrated via a **state machine** (LangGraph):

```
┌─────────────┐
│ CLASSIFY │ ← Determine which pattern to use
└──────┬──────┘
 │
 ▼
┌─────────────┐
│ ROUTE │ ← Direct to single-pass, reflection, debate, or hierarchical
└──────┬──────┘
 │
 ▼
┌─────────────┐
│ EXECUTE │ ← Run selected pattern
└──────┬──────┘
 │
 ▼
┌─────────────┐
│ VERIFY │ ← Validate output, check for hallucinations
└──────┬──────┘
 │
 ▼
 [response]
```

Each pattern becomes a **node** in the graph. LangGraph handles state transitions, retries, and monitoring.

In [ ]:
def classify_node(state: ConversationState):
    """
    TODO #9: Implement `classify_node()`.

    Steps:
    1. Set up: ─
    2. Call `ConversationState()` to produce the result
    3. Define helper function `classify_node()`
    4. Define helper function `route_node()`
    5. Define helper function `execute_node()`
    6. Call `execution()` to produce the result
    7. Process data
    8. Define helper function `verify_node()`
    9. Call `verification()` to produce the result
    10. Process data
    11. Compute `query`
    12. Process data
    13. Compute `state`
    14. Process data
    15. Call `alerting()` to produce the result

    Hint:
    # implement using the APIs described above

    Returns: state
    """
    raise NotImplementedError("TODO: implement classify_node()")

def route_node(state: ConversationState):
    """
    TODO #9: Implement `route_node()`.

    Steps:
    1. Set up: ─
    2. Call `ConversationState()` to produce the result
    3. Define helper function `classify_node()`
    4. Define helper function `route_node()`
    5. Define helper function `execute_node()`
    6. Call `execution()` to produce the result
    7. Process data
    8. Define helper function `verify_node()`
    9. Call `verification()` to produce the result
    10. Process data
    11. Compute `query`
    12. Process data
    13. Compute `state`
    14. Process data
    15. Call `alerting()` to produce the result

    Hint:
    # implement using the APIs described above

    Returns: state
    """
    raise NotImplementedError("TODO: implement route_node()")

def execute_node(state: ConversationState):
    """
    TODO #9: Implement `execute_node()`.

    Steps:
    1. Set up: ─
    2. Call `ConversationState()` to produce the result
    3. Define helper function `classify_node()`
    4. Define helper function `route_node()`
    5. Define helper function `execute_node()`
    6. Call `execution()` to produce the result
    7. Process data
    8. Define helper function `verify_node()`
    9. Call `verification()` to produce the result
    10. Process data
    11. Compute `query`
    12. Process data
    13. Compute `state`
    14. Process data
    15. Call `alerting()` to produce the result

    Hint:
    # implement using the APIs described above

    Returns: state
    """
    raise NotImplementedError("TODO: implement execute_node()")

def verify_node(state: ConversationState):
    """
    TODO #9: Implement `verify_node()`.

    Steps:
    1. Set up: ─
    2. Call `ConversationState()` to produce the result
    3. Define helper function `classify_node()`
    4. Define helper function `route_node()`
    5. Define helper function `execute_node()`
    6. Call `execution()` to produce the result
    7. Process data
    8. Define helper function `verify_node()`
    9. Call `verification()` to produce the result
    10. Process data
    11. Compute `query`
    12. Process data
    13. Compute `state`
    14. Process data
    15. Call `alerting()` to produce the result

    Hint:
    # implement using the APIs described above

    Returns: state
    """
    raise NotImplementedError("TODO: implement verify_node()")

## 9 · Monitoring and Observability

In production, you need to monitor:

1. **Reflection loop metrics**
 - How many iterations to converge? (avg 2.3, alert if >5)
 - What % of queries need reflection? (target: 15–20%)

2. **Debate metrics**
 - Did agents reach consensus? (target: >90%)
 - Which agent wins most often? (monitor for bias)

3. **Hierarchical orchestration**
 - How many batches per complex order? (avg 3.2)
 - Worker failure rate? (target: <1%)

4. **Tool selection**
 - Which fallback tier is used? (cache: 70%, DB: 25%, API: 5%)
 - How many retries before escalation? (avg 1.8)

Let's simulate a monitoring dashboard.

In [ ]:
# TODO: Implement this cell
#  (─)
#
# Steps:
# 1. Set up: ─
# 2. Process data
# 3. Compute `days` using `data()`
# 4. Compute `reflection_iterations`
# 5. Compute `debate_consensus`
# 6. Compute `tool_cache` using `distribution()`
# 7. Compute `pattern_single`
# 8. Plot results -- call `use()`
# 9. Plot results -- call `set_facecolor()`
# 10. Plot results -- call `plot()`
# 11. Plot results -- call `stackplot()`
# 12. Plot results -- call `array()`
# 13. Call `set_xlabel()` to produce the result
# 14. Plot results -- call `tight_layout()`
# 15. Call `mean()` to produce the result
# 16. Process data
#
# Hint:
#    days = np.arange(???)
#    axes = plt.subplots(???)
#    x = np.arange(???)
#    bottom = np.array(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ──────────────────────────────────────────────────────────────
# Monitoring Dashboard (Simulated Data)
# ──────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import numpy as np

# Simulated monitoring data (7 days)
days = np.arange(1, 8)

# Reflection loop convergence
reflection_iterations = [2.1, 2.4, 2.2, 2.5, 2.3, 2.6, 2.4]  # avg iterations
reflection_over_5 = [0.02, 0.03, 0.02, 0.04, 0.03, 0.05, 0.03]  # % taking >5 iterations

# Debate consensus rate
debate_consensus = [92, 94, 91, 93, 95, 92, 94]  # % reaching consensus

# Tool fallback distribution (% of queries)
tool_cache = [72, 70, 68, 71, 69, 70, 68]
tool_db = [23, 25, 26, 24, 26, 25, 27]
tool_api = [5, 5, 6, 5, 5, 5, 5]

# Pattern distribution
pattern_single = [65, 63, 64, 62, 63, 64, 65]
pattern_reflection = [18, 20, 19, 21, 20, 19, 18]
pattern_debate = [8, 7, 8, 7, 8, 7, 8]
pattern_hierarchical = [6, 7, 6, 7, 6, 7, 6]
pattern_tool = [3, 3, 3, 3, 3, 3, 3]

# Create monitoring dashboard
plt.style.use('dark_background')
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('SmartPizzaBot Production Monitoring Dashboard', 
             fontsize=16, fontweight='bold', y=0.995)

for ax in axes.flat:
    ax.set_facecolor('#1a1a2e')

# Plot 1: Reflection convergence
ax1 = axes[0, 0]
ax1.plot(days, reflection_iterations, 'o-', color='#4a9eff', linewidth=2, markersize=8, label='Avg iterations')
ax1.axhline(y=5.0, color='#ff6b6b', linestyle='--', linewidth=2, label='Alert threshold (>5)')
ax1.fill_between(days, 2, 5, alpha=0.1, color='#51cf66')
ax1.set_xlabel('Day', fontsize=11)
ax1.set_ylabel('Iterations to converge', fontsize=11)
ax1.set_title('Reflection Loop Convergence', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.2)
ax1.set_ylim(0, 6)

# Plot 2: Debate consensus rate
ax2 = axes[0, 1]
ax2.plot(days, debate_consensus, 's-', color='#51cf66', linewidth=2, markersize=8, label='Consensus rate')
ax2.axhline(y=90, color='#ffd93d', linestyle='--', linewidth=2, label='Target (>90%)')
ax2.fill_between(days, 90, 100, alpha=0.1, color='#51cf66')
ax2.set_xlabel('Day', fontsize=11)
ax2.set_ylabel('Consensus rate (%)', fontsize=11)
ax2.set_title('Debate Pattern Consensus', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.2)
ax2.set_ylim(85, 100)

# Plot 3: Tool fallback distribution
ax3 = axes[1, 0]
ax3.stackplot(days, tool_cache, tool_db, tool_api, 
              labels=['Cache (free)', 'Database ($0.0001)', 'API ($0.001)'],
              colors=['#51cf66', '#4a9eff', '#ff6b6b'], alpha=0.8)
ax3.set_xlabel('Day', fontsize=11)
ax3.set_ylabel('Distribution (%)', fontsize=11)
ax3.set_title('Tool Fallback Distribution', fontsize=12, fontweight='bold')
ax3.legend(loc='upper right', fontsize=9)
ax3.grid(alpha=0.2)
ax3.set_ylim(0, 100)

# Plot 4: Pattern distribution
ax4 = axes[1, 1]
width = 0.6
x = np.arange(len(days))
ax4.bar(x, pattern_single, width, label='Single-pass', color='#666')
ax4.bar(x, pattern_reflection, width, bottom=pattern_single, 
        label='Reflection', color='#4a9eff')
ax4.bar(x, pattern_debate, width, 
        bottom=np.array(pattern_single)+np.array(pattern_reflection),
        label='Debate', color='#ff6b6b')
ax4.bar(x, pattern_hierarchical, width,
        bottom=np.array(pattern_single)+np.array(pattern_reflection)+np.array(pattern_debate),
        label='Hierarchical', color='#51cf66')
ax4.bar(x, pattern_tool, width,
        bottom=np.array(pattern_single)+np.array(pattern_reflection)+np.array(pattern_debate)+np.array(pattern_hierarchical),
        label='Tool Selection', color='#ffd93d')

ax4.set_xlabel('Day', fontsize=11)
ax4.set_ylabel('Distribution (%)', fontsize=11)
ax4.set_title('Pattern Usage Distribution', fontsize=12, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels([f'D{d}' for d in days])
ax4.legend(fontsize=9, loc='upper right')
ax4.set_ylim(0, 100)

plt.tight_layout()
plt.savefig('img/monitoring_dashboard.png', dpi=150, facecolor='#1a1a2e')
print("✓ Monitoring dashboard saved to img/monitoring_dashboard.png")
plt.show()

# Key metrics summary
print("\n" + "=" * 70)
print("7-DAY METRICS SUMMARY")
print("=" * 70)
print(f"Reflection convergence:   {np.mean(reflection_iterations):.2f} avg iterations (target: <3.0)")
print(f"Reflection >5 iterations: {np.mean(reflection_over_5)*100:.1f}% (alert threshold: >5%)")
print(f"Debate consensus rate:    {np.mean(debate_consensus):.1f}% (target: >90%)")
print(f"Tool cache hit rate:      {np.mean(tool_cache):.1f}% (target: >65%)")
print(f"Pattern distribution:")
print(f"  Single-pass:     {np.mean(pattern_single):.1f}%")
print(f"  Reflection:      {np.mean(pattern_reflection):.1f}%")
print(f"  Debate:          {np.mean(pattern_debate):.1f}%")
print(f"  Hierarchical:    {np.mean(pattern_hierarchical):.1f}%")
print(f"  Tool Selection:  {np.mean(pattern_tool):.1f}%")

print("\n✓ All metrics within target ranges — system healthy")

## Summary — What You Learned

You now know **four production agentic patterns** that unlock iterative refinement:

1. **Reflection Loop** — Draft → Critique → Revise
 - Use for: Contradictory inputs, edge cases
 - Cost: 2.9× baseline → 6× error reduction

2. **Multi-Agent Debate** — Propose → Challenge → Vote
 - Use for: High-stakes decisions, policy compliance
 - Cost: 6× baseline → 8× error reduction

3. **Hierarchical Orchestration** — Planner → Workers → Verifier
 - Use for: Complex multi-step tasks
 - Cost: 5× baseline → 15× error reduction

4. **Tool Selection with Fallback** — Meta-agent + retry chains
 - Use for: Unreliable tools, cost optimization
 - Cost: 1.5× baseline → 4× error reduction

**Production deployment checklist:**
- Route queries through pattern classifier
- Orchestrate with LangGraph state machine
- Monitor convergence, consensus, tool usage
- Set budget alerts (cost per conversation)
- A/B test single-pass vs reflection (50/50 split)

**Next:** [Azure Supplement](notebook_supplement.ipynb) — Deploy these patterns to production with Azure Container Instances, monitor with Application Insights, and run A/B tests.